# IsoFormer reproduction: Enformer + NT + ESM-2

Can genomic context, a spliced transcript and its protein sequence jointly improve prediction of that transcript's expression across tissues? A gene can produce different transcript versions, or **isoforms**. Genomic DNA supplies surrounding sequence context; the spliced RNA supplies the particular transcript, including its untranslated regions; a supplied protein sequence represents its translated amino-acid chain. These inputs describe related but different parts of the same transcript's biological context.

GTEx supplies **30 expression targets for each transcript**, one per tissue. The released targets represent tissue-averaged transcript TPM transformed with `log(1 + x)`. The prediction concerns RNA abundance across tissues; it is not a measurement of protein production. The [GTEx target explanation](../../docs/methods/data-and-targets.md#gtex-tissue-expression-labels) traces those measurements and transformations.

![Authors' Figure 2: RNA isoforms and their proteins, followed by the encoder-to-expression prediction framework.](figures/isoformer_paper_figure2.png)

**From biology to prediction.** Panel A illustrates isoforms and their protein products; panel B connects sequence encoders, a fusion module and expression prediction. The figure illustrates protein-producing transcripts; our agreed input policy also retains otherwise eligible transcripts without supplied proteins. Original artwork from Garau-Luis et al., [Figure 2, p. 5](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf#page=5), cropped from the published PDF without redrawing.

This notebook prepares a faithful reconstruction of a released IsoFormer configuration so we can test what adding RNA and protein contributes, and whether its fusion operation earns its complexity. It contains fusion and input-checking code; pretrained encoder integration and training remain unfinished. The [earlier frozen GTEx pilot](tier2_lucaone_isoformer.ipynb) uses cached embeddings and a different local aggregator. Its results remain evidence for that pilot, rather than results for the reconstruction here.


## Which reference are we reconstructing?

“Enformer reference” names the **DNA branch of a three-encoder IsoFormer**, not the whole model. The selected [released configuration](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/config.json) combines:

| Input for one transcript | Encoder | Information supplied |
| --- | --- | --- |
| Genomic DNA window | Enformer | Sequence around the transcript's genomic location |
| Spliced transcript RNA | Nucleotide Transformer (NT v2 250M) | The transcript sequence, represented in NT's nucleotide alphabet |
| Supplied protein sequence | ESM-2 150M | The translated amino-acid sequence, when available |

The sources answer different questions. The **pinned released code** defines the computation we are reconstructing. The **final paper** describes the biological task, training settings and comparisons. **Official inference examples** clarify loading and input preparation, but do not recover the missing trainer. Where those accounts disagree, this notebook names the selected setting and its reconstruction assumption; a newer repository revision alone does not resolve the scientific discrepancy. The [specification](RUNBOOK.md#released-behavior-retained--november-9) retains the source/version trace.

The agreed order is **Enformer first, NT-DNA next, Borzoi later**, keeping the RNA NT and protein ESM-2 branches fixed. NT and ESM-2 are therefore already included; the next stage adds NT in the DNA role. The paper evaluates these alternative DNA backbones and seven modality combinations, while only the released Enformer all-three fusion foundations are implemented here. [Paper, Table 3 and Appendix D.2/Table 8](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf).

Read the notebook as architecture → source settings and decisions → eligible transcripts and prepared inputs → model construction → comparisons. The [GTEx audit](tier2_isoformer_gtex_audit.ipynb) measures what the released dataset supplies; this notebook explains how those observations constrain the model and its future training.


## How the released fusion works

![Authors' Figure 1: successive cross-attention and residual connections update DNA, RNA and protein representations.](figures/isoformer_paper_figure1.png)

**Reading the paper's figure.** Follow each column from bottom to top. Cross-attention brings in information from another representation; each plus sign adds the resulting update to the incoming representation, a **residual connection**. The paper shows updates to all three streams. The selected released configuration follows the **left-hand DNA pathway**: RNA contributes first, then protein. Original artwork from Garau-Luis et al., [Figure 1, p. 2](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf#page=2), cropped from the published PDF without redrawing.

In the released path, DNA embeddings supply **queries**, which are compared with RNA **keys** to determine attention weights. Those weights combine RNA **values**, and a learned projection converts the combination into an update added to DNA. The next block uses the updated DNA to query protein. RNA and protein provide information to these fusion blocks; the blocks do not update those two streams. This ordering describes a model computation, not DNA physically consulting RNA or protein, and its weights are not measured regulatory interactions. [Released forward](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py).

**From many positions to 30 predictions.** Pooling averages the five updated DNA bins selected by `765:770`, producing one vector of width 3,072. The prediction head maps it through `3072 → 6144 → 30`, with Softplus between the two linear layers. The 30 outputs correspond to the tissue targets. We retain those released indices rather than substituting a near-center window; they must not be described as central or TSS-aligned. The [pooling-coordinate trace](RUNBOOK.md#released-computation-and-proposed-notebook) explains that limitation.

The selected fusion operator includes attention, an output projection and residual addition, without an additional fusion LayerNorm, feed-forward block or active attention dropout. This is more specific than a generic Transformer block. NT-DNA and Borzoi will need their own pooling and head specifications; these coordinates and widths belong to the Enformer configuration. [Released attention operator](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_esm.py).


---

## Setup

Use the [root notebook environment instructions](../../README.md#notebook-dependencies). Prepare project imports and configuration before reviewing the source settings and inspecting inputs.


In [ ]:
# Named experiment settings and their status fields.
from dataclasses import dataclass, fields

# Filesystem paths and the import search path.
from pathlib import Path
import hashlib  # Verify the small hg38 chromosome-size input before using it.
import sys

# Tabular inspection; load NumPy before PyTorch in the Windows environment.
import numpy as np
import pandas as pd

# Neural-network definitions and available hardware, without creating a model.
import torch

# Render notebook tables.
from IPython.display import display

### Project paths and shared code

Shared modules hold the tested model and input helpers. Experiment settings and status reporting stay in this notebook, beside the choices they describe.

In [ ]:
# Walk upward from the working directory so root and stage-folder launches work.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "Code" / "shared_code").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Open the notebook within the project checkout.")

# Add Code to the search path so shared_code imports resolve to this checkout.
CODE_DIR = REPO_ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Tissue order is shared with the dataset registry; no dataset is loaded here.
from shared_code.datasets import GTEX_TISSUES
# Display source links/definitions without calling the referenced functions.
from shared_code.notebook import show_source
from shared_code.fusion import isoformer_inputs as inputs
from shared_code.fusion import isoformer_reference as reference
from shared_code.fusion import isoformer_adapters as adapters

### Settings and deliberate boundaries

The settings below apply to **Enformer DNA + NT RNA + ESM-2 protein**. They do not switch between the later DNA-backbone comparisons.

Agreed sequence, DNA-boundary and raw-partition policies have explicit values. Remaining decisions stay unset, and training still refuses because the pretrained data/training route is unfinished. Optional local CSV and verified hg38 chromosome-size paths permit preparation in memory; neither enables a download or writes results.


In [ ]:
# False shows source links; True displays the maintained definitions in full.
SHOW_IMPLEMENTATION = False
# Training is not implemented; True raises at the guard below.
RUN_TRAINING = False
# None skips data access; a path permits schema and sequence-policy preparation.
LOCAL_GTEX_CSV = None  # Original GTEx_final.csv, not the pilot subset.
LOCAL_CHROMOSOME_SIZES = None  # Local hg38.chrom.sizes; no genome download.
# Same verified hg38 metadata used by the saved GTEx audit.
CHROMOSOME_SIZES_SHA256 = (
    "e1d9152418038457a959e949d99c9caf7ae3e4f87cfbb4ffe7d8b9a54ba1202b"
)

@dataclass(frozen=True)
class ReproductionChoices:
    """Agreed input settings are explicit; remaining choices stay None.

    Declaring a value does not verify the real pretrained pipeline. Normalization
    and training settings remain separate from the implemented sequence policies.
    """

    pool_window: tuple[int, int] = (765, 770)
    dna_input_nucleotides: int = 196_608
    dna_extra_filter_nucleotides: int = 196_608
    dna_output_bins: int | None = None  # Verify the actual pretrained output.
    protein_mask_policy: str = "released_unmasked"
    missing_protein_policy: str = "bypass_projected_update"
    rna_max_nucleotides: int = 12_288  # Retain the 3′ suffix; no second token crop.
    protein_max_characters: int = 1_200  # Source string, before tokenizer markers.
    protein_crop: str = "exclude"  # Exclude long supplied proteins; never recode as absent.
    raw_eval_policy: str = "exclude"
    target_normalization: str | None = None
    validation_sampling_unit: str | None = None
    validation_seed: int | None = None
    cohort_boundary_policy: str = "released_example"
    optimizer_options: str | None = None
    scheduler: str | None = None
    batch_loss_reduction: str | None = None
    precision: str | None = None
    stopping_rule: str | None = None


choices = ReproductionChoices()

# Name a future destination without creating a directory or writing results.
REPRODUCTION_RESULTS = CODE_DIR / "stage4" / "results" / "isoformer_enformer"


In [ ]:
# Explanations used by the local status tables and the training refusal.
DECISION_REASONS = {
    "pool_window": "Agreed literal released bins; do not silently recenter them.",
    "dna_input_nucleotides": "Agreed Enformer genomic context, before encoding.",
    "dna_extra_filter_nucleotides": "Retain the example's shifted extra-filter length.",
    "dna_output_bins": "Verify actual Enformer output and cropping for the pinned dependency.",
    "protein_mask_policy": "Agreed omission of the protein fusion mask; retain its limitation.",
    "missing_protein_policy": "Skip the complete projected update for absent proteins.",
    "rna_max_nucleotides": "Agreed 3′ suffix from Table 6; no second token-count crop.",
    "protein_max_characters": "Agreed source-character ceiling, before tokenizer markers.",
    "protein_crop": "Exclude long supplied proteins from every matched model's cohort.",
    "raw_eval_policy": "Agreed exclusion from fitting, validation and reported test evaluation.",
    "target_normalization": (
        "Per-tissue normalization formula, fitting population and inverse remain "
        "unresolved."
    ),
    "validation_sampling_unit": (
        "Choose transcript or gene sampling; the paper does not recover exact "
        "membership."
    ),
    "validation_seed": (
        "Record a reproducible seed; validation varies between repeats as the paper "
        "specifies."
    ),
    "cohort_boundary_policy": (
        "Retain chrMT exclusion, strict right boundary and the shifted extra filter."
    ),
    "optimizer_options": "Adam options beyond the reported learning rate were not recovered.",
    "scheduler": "No complete original schedule was recovered; do not inherit the BLF scheduler.",
    "batch_loss_reduction": (
        "The paper sums tissue errors; batch reduction still needs an explicit rule."
    ),
    "precision": "Declare and verify numerical precision on the selected hardware.",
    "stopping_rule": (
        "Declare validation monitor, frequency and stopping rule; no epoch/patience "
        "default is adopted."
    ),
}

# Declared settings cannot bypass missing real-model integration or a missing trainer.
IMPLEMENTATION_BLOCKERS = (
    (
        "Concrete adapters exist, but original-pretrained loading, tokenizer/asset "
        "identity and real-model compatibility are not verified."
    ),
    (
        "Reference eligibility is implemented; real CSV/genome integration, target "
        "transformation and evaluation-scale implementation remain incomplete."
    ),
    (
        "Real-data batching must preserve the chosen per-example protein padding; "
        "the preparation helper refuses unequal padded lengths."
    ),
    (
        "The reproduction trainer, checkpoint recovery and result writer are not "
        "implemented."
    ),
)


In [ ]:
def decision_rows(choices):
    """Display unresolved or declared choices without claiming they are validated."""
    if not isinstance(choices, ReproductionChoices):
        raise TypeError("choices must be ReproductionChoices")
    return [
        {
            "setting": field.name,
            "value": getattr(choices, field.name),
            "status": (
                "unresolved" if getattr(choices, field.name) is None
                else "declared, not validated"
            ),
            "reason": DECISION_REASONS[field.name],
        }
        for field in fields(choices)
    ]


def training_blockers(choices):
    """List scientific decisions and the remaining implementation prerequisites."""
    decisions = [
        f"{row['setting']}: {row['reason']}"
        for row in decision_rows(choices)
        if row["status"] == "unresolved"
    ]
    return decisions + list(IMPLEMENTATION_BLOCKERS)


# Refuse training before hardware reporting or the optional CSV read.
if RUN_TRAINING:
    blockers = training_blockers(choices)
    raise RuntimeError(
        "IsoFormer reproduction training is unavailable in this implementation batch:\n- "
        + "\n- ".join(blockers)
    )

In [ ]:
# Report backend availability; no training device is selected or benchmarked.
if torch.cuda.is_available():
    available_device = "CUDA: " + torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    available_device = "Apple MPS"
else:
    available_device = "CPU"

print("Available hardware:", available_device)
print("No model is allocated. Runtime and peak memory have not been measured.")

---

## Recovered settings and unresolved choices

The next table lists settings reported by the paper, including the learning rate, batch size and validation fraction. It is a source summary, not a complete training configuration. A listed RNA length, for example, does not resolve the paper's conflicting token limits or apply a crop.

The paper changes **validation membership and initialization across repeats**. Its early-stopping description does not supply every setting needed for our trainer. Exact seed IDs, the complete stopping rule and remaining optimizer details must therefore be explicit reconstruction choices. The earlier 30-epoch cap and five-epoch patience were unsupported local proposals and are not adopted. [Paper, Section 5.1 and Appendix B](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf).


In [ ]:
# Reported values only; this table does not configure or start a fit.
PAPER_SETTINGS = {
    "optimizer": "Adam",
    "learning_rate": 3e-5,
    "batch_size": 64,
    "validation_fraction": 0.05,
    "validation_membership": "varies with each training repeat",
    "dna_nucleotides": 196_608,
    "rna_nucleotides": 12_288,
    "rna_retained_end": "3prime",
}

display(pd.DataFrame(
    PAPER_SETTINGS.items(), columns=["paper setting", "reported value"]
).set_index("paper setting"))

### What remains a decision?

**Read the next table as implementation state.** Agreed policies have declared values; `unresolved` means a field still needs a decision or actual-backbone verification. A declared setting is not evidence that real pretrained models have been checked.

DNA eligibility now follows the released example's 196,608-base input and shifted filter, with `chrMT` excluded. Raw `eval` rows remain unused. Validation sampling, target normalization and the remaining training recipe are still open. Token mappings and actual encoder output shapes require engineering verification rather than an arbitrary choice. The [input-policy specification](RUNBOOK.md#agreed-cohort-and-sequence-policies) distinguishes source-backed rules from reconstruction assumptions.


In [ ]:
display(pd.DataFrame(decision_rows(choices)).set_index("setting"))

---

## Check the cohort without changing targets

Here, the **cohort** means the GTEx transcript rows admitted to a particular training and evaluation comparison. Each row carries transcript/gene identifiers, genomic coordinates, RNA, any supplied protein, a raw partition label and the 30 tissue targets. Checking that these fields are usable is separate from deciding which rows to admit.

**Agreed population.** Retain otherwise eligible transcripts without supplied proteins: their RNA-expression targets are still meaningful. Record protein availability separately from transcript annotation. Exclude rows with supplied proteins longer than 1,200 source characters from every model in the matched comparison, so a DNA-only model is not evaluated on a different population merely because it does not read protein. A protein-only comparison additionally needs the supplied-protein subset, with its competitors evaluated on those same rows. The [policy record](RUNBOOK.md#agreed-cohort-and-sequence-policies) distinguishes the paper's numerical limit from our exclusion choice.

**Limits of this inspection.** Schema validation accepts absent proteins and the released raw `train`, `test` and `eval` labels without assigning them a new role. The sequence-policy step below then applies only the agreed crop/exclusion rules. The later eligibility step applies the agreed DNA-boundary and raw `eval` exclusions. Passing schema or sequence checks alone does not establish the final training population.


In [ ]:
cohort = None
if LOCAL_GTEX_CSV is None:
    print("No cohort opened. Supply an original local CSV for in-memory preparation.")
else:
    local_csv = Path(LOCAL_GTEX_CSV).expanduser().resolve(strict=True)
    supplied_rows = pd.read_csv(local_csv)
    # Preserve raw partitions and missing proteins here; eligibility is a separate step.
    cohort = inputs.validate_cohort(supplied_rows, GTEX_TISSUES)
    print("Schema checked; these are not yet the eligible reproduction rows.")
    # Inspect supplied partition counts and target ranges without normalization.
    display(cohort.rows.groupby("split").size().rename("supplied rows").to_frame())
    display(pd.DataFrame({
        "tissue": cohort.tissues,
        "minimum supplied target": cohort.labels.min(axis=0),
        "maximum supplied target": cohort.labels.max(axis=0),
    }).set_index("tissue"))


**Preserve the meaning of the targets.** The released labels are described as already log-transformed, so applying a second log would change their meaning. Additional normalization would rescale those targets for training; it is a separate operation. The release exposes normalization statistics, but the historical formula, fitting population and conversion back to the evaluation scale remain unresolved. The current checks preserve the supplied values and fit no transformation. [Dataset card](https://huggingface.co/datasets/InstaDeepAI/multi_omics_transcript_expression/blob/330ab1795abca683e692c7ee138a2e89c0c882f1/README.md) · [target provenance](../../docs/methods/data-and-targets.md#gtex-tissue-expression-labels).


In [ ]:
show_source(inputs.validate_cohort, full=SHOW_IMPLEMENTATION)

## Prepare sequences before assigning validation

Eligibility depends partly on whether each transcript can supply the requested genomic context. The DNA input is a window from an hg38 chromosome around the transcript's strand-dependent start: `start - 1` on the positive strand and `end - 1` on the negative strand. Negative-strand windows are reverse-complemented to match transcript orientation. This genomic window differs from the spliced RNA supplied in the same row.

The helper preserves two released checks. The **actual-window check** rejects a slice extending beyond the chromosome, including the release's strict rejection when its right endpoint equals chromosome length. The **extra filter** checks an additional interval around the window's already shifted left edge. This is not a symmetric check around the transcript start. Neither check pads or shortens rejected DNA. [Pinned loader](https://huggingface.co/datasets/InstaDeepAI/multi_omics_transcript_expression/blob/330ab1795abca683e692c7ee138a2e89c0c882f1/multi_omics_transcript_expression.py).

**Adopted reconstruction.** Use 196,608 bases for both the DNA input and extra filter, preserving the official inference example's behavior and the release's `chrMT` exclusion. Applying those example settings to training is our explicit reconstruction; it does not establish the authors' original training membership. The [GTEx audit](tier2_isoformer_gtex_audit.ipynb) measures the example's effect across the raw table. [Official inference example](https://github.com/instadeepai/nucleotide-transformer/blob/123c50144df3a5332bc96593da7e35f32b394595/notebooks/isoformer/inference.ipynb).

Window eligibility needs chromosome lengths, not the full genome. After the RNA/protein rules below, the notebook applies this policy using a supplied, verified hg38 chromosome-size file. Obtaining genome sequence and connecting it to the real pretrained encoders remain separate work.


In [ ]:
show_source(
    inputs.released_window,
    inputs.extract_released_window,
    inputs.apply_reference_eligibility,
    full=SHOW_IMPLEMENTATION,
)

### Transcript and protein context

**RNA: retain the last 12,288 nucleotides.** Keep shorter transcripts unchanged; for longer ones, remove excess sequence from the 5′ end and retain the 3′ end. The paper motivates that end by regulatory context relevant to RNA abundance. The number follows Table 6, while Table 1 gives 12,282 bases and Table 7 gives 2,048 tokens. Prioritizing Table 6 is our declared reconstruction choice. The released inference path does not apply this crop. [Paper, Section 4.3 and Tables 1, 6–7](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf).

A nucleotide count and a token count measure different things. For example, 12,288 uppercase A/C/G/T bases form 2,048 sixmers plus a starting CLS token: **2,049 tokens**. We will verify actual token counts and adapter compatibility without silently adding a second crop at 2,048 tokens. The sequence-policy helper below applies that suffix to the optional local table, while preserving supplied targets and source-row identities.

**Protein: retain complete supplied strings up to 1,200 characters.** The numerical limit comes from Table 6's amino-acid maximum; choosing exclusion rather than cropping is our reconstruction because the historical admission/crop rule was not recovered. A longer supplied protein excludes its row from the common comparison cohort; it must not be converted into an absent protein. This limits conclusions about transcripts encoding longer proteins. [Paper, Table 6](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf).

Count this protein threshold before special tokens or padding. Ordinary residue letters each count as one character; special symbols need explicit tokenizer checks, rather than silent cleaning. Other paper/release context limits do not by themselves specify which rows the authors admitted.

**Padding adds artificial positions; cropping removes sequence.** The released tokenizer pads short inputs to 1,024 tokens without enabling truncation. It therefore does not impose a 1,024-token maximum. The protein encoder receives a mask, but the selected released fusion step omits its protein mask, allowing padded positions to contribute there. Future batching must preserve that per-example behavior; adding more padding is consequential. This retained limitation is separate from bypassing genuinely absent proteins. [Released tokenizer](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/isoformer_tokenizer.py) · [masking decision](RUNBOOK.md#released-behavior-retained--november-9).


In [ ]:
show_source(
    inputs.crop_rna_3prime,
    inputs.apply_sequence_policies,
    full=SHOW_IMPLEMENTATION,
)


In [ ]:
# Apply sequence rules first; DNA and raw-partition eligibility follow below.
prepared_cohort = None
sequence_policy_report = None
if cohort is not None:
    if (choices.protein_crop != "exclude"
            or choices.missing_protein_policy != "bypass_projected_update"):
        raise ValueError("This preparation implements the agreed exclusion/bypass policies only.")
    prepared_cohort, sequence_policy_report = inputs.apply_sequence_policies(
        cohort,
        rna_max_nucleotides=choices.rna_max_nucleotides,
        protein_max_characters=choices.protein_max_characters,
    )
    display(sequence_policy_report.groupby(
        ["split", "protein_present", "sequence_eligible"], dropna=False,
    ).size().rename("transcripts").to_frame())
    print("Sequence rules applied in memory; DNA and partition eligibility follow below.")


### Apply the agreed cohort eligibility

The sequence-policy report covers the original table. The next report covers **only rows retained by those RNA/protein rules**, then excludes raw `eval`, `chrMT` and rejected DNA windows. Retained rows keep their source indices, target values and raw train/test labels. These are shared comparison rows, regardless of which modalities a later model reads.

Supply a local [UCSC hg38 chromosome-size file](https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes) matching the checksum used by the [GTEx audit](tier2_isoformer_gtex_audit.ipynb). Lengths suffice to check windows; this step does not load genomic DNA. A missing file setting leaves eligibility unevaluated. A mismatched file, duplicate chromosome, or missing length for a candidate train/test row stops preparation rather than silently narrowing the population.

The report retains separate main-window and extra-filter flags where DNA was checked. Its single reason follows raw `eval` → mitochondrial exclusion → the release's extra-first boundary order; skipped rows have no DNA assessment. The audit's summary uses main-before-extra categories across the raw table, so its reason counts are not directly comparable to this sequential report. Neither reporting order changes which windows pass.


In [ ]:
def read_verified_chromosome_sizes(path, expected_sha256):
    """Read local hg38 length metadata; never fetch a genome or repair the file."""
    path = Path(path).expanduser().resolve(strict=True)
    payload = path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError("Chromosome sizes do not match the declared hg38 source.")

    sizes = {}
    for line_number, line in enumerate(payload.decode("utf-8").splitlines(), 1):
        parts = line.split()
        if len(parts) != 2:
            raise ValueError(f"Expected chromosome and length on line {line_number}.")
        name, text_length = parts
        if name in sizes:
            raise ValueError(f"Duplicate chromosome name: {name}")
        length = int(text_length)
        if length <= 0:
            raise ValueError(f"Chromosome length must be positive: {name}")
        sizes[name] = length
    if not sizes:
        raise ValueError("Chromosome-size file is empty.")
    return sizes


In [ ]:
# Keep incomplete preparation distinct from a cohort admitted for later splitting.
eligible_cohort = None
eligibility_report = None
if prepared_cohort is None:
    print("No sequence-prepared cohort; eligibility was not evaluated.")
elif LOCAL_CHROMOSOME_SIZES is None:
    print("Set LOCAL_CHROMOSOME_SIZES to verified hg38 metadata to check eligibility.")
else:
    if choices.cohort_boundary_policy != "released_example":
        raise ValueError("This step implements the agreed released-example boundary policy.")
    chromosome_sizes = read_verified_chromosome_sizes(
        LOCAL_CHROMOSOME_SIZES, CHROMOSOME_SIZES_SHA256,
    )
    eligible_cohort, eligibility_report = inputs.apply_reference_eligibility(
        prepared_cohort,
        chromosome_sizes,
        sequence_length=choices.dna_input_nucleotides,
        extra_filter_length=choices.dna_extra_filter_nucleotides,
        raw_eval_policy=choices.raw_eval_policy,
    )
    outcomes = eligibility_report.assign(
        outcome=eligibility_report["reference_exclusion"].fillna("retained"),
    )
    display(outcomes.groupby(["split", "outcome"]).size().rename("transcripts").to_frame())
    print("Eligibility applied in memory. No validation split or training was performed.")


### Validation comes only from training

Once eligible rows are known, validation selects training settings/checkpoints, while the supplied test partition measures final performance. The paper takes 5% from training and changes validation membership across repeats. The released dataset loader instead maps its named validation and test requests to the same underlying test rows. We do not use that alias as independent validation; it does not establish how the authors actually trained. [Paper, Sections 4.4–5.1](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf) · [dataset loader](https://huggingface.co/datasets/InstaDeepAI/multi_omics_transcript_expression/blob/330ab1795abca683e692c7ee138a2e89c0c882f1/multi_omics_transcript_expression.py).

**Raw `eval` remains unused.** The eligibility step excludes those rows from training, validation and reported test evaluation without changing their source labels. This is our declared reconstruction from the paper's train-derived validation and the loader's exposed train/test splits; it is not a recovered historical use of `eval`. Remaining test rows keep their original membership.

The helper offers **transcript sampling**, where sibling isoforms from one gene may be divided between training and validation, or **gene sampling**, where all transcripts from a selected gene move together. With gene sampling, 5% of genes need not equal 5% of transcript rows. The sampling unit and seed remain open. The implemented NumPy permutation and upward rounding are reconstruction details, not recovered historical settings.

Pass `eligible_cohort` to the eventual validation step, after both sequence and DNA/partition policies. The validator still refuses any raw `eval` rows if an earlier unfiltered object is passed. `gene_overlap` reports shared gene IDs without changing partitions; it does not remove overlap with supplied test rows. The calls below display definitions only, so no validation set is assigned here.


In [ ]:
show_source(
    inputs.validation_partitions,
    inputs.gene_overlap,
    full=SHOW_IMPLEMENTATION,
)

---

## Connect the live encoders to the released fusion

With inputs and partitions defined, each encoder must supply the representations consumed by the fusion operator. `ReferenceFusion` already implements RNA and then protein updates to DNA, selected-DNA mean pooling and the tissue head. It requires explicit pooling and protein-mask arguments; integration must supply the agreed `765:770` and `released_unmasked` choices.

The local implementation makes two compatibility changes explicit: RNA masks broadcast per batch member, and pooling uses a direct mean slice instead of the release's CPU-created pooling mask. The optional `mask_padding` protein-fusion alternative changes behavior and is outside the selected reference. Actual backbone integration must still verify tensor shapes and mask alignment.

`LiveIsoFormer` joins three supplied **adapters**—interfaces that convert each encoder's output into the token/bin embeddings this fusion expects. It does not freeze parameters or detach their tensors, so gradients can reach the encoders during future training. Concrete Enformer and token-encoder adapters are prepared below, but original pretrained weights are not loaded. Synthetic tests use tiny substitute encoders to check the adapter calls and gradient connection; they do not establish pretrained compatibility or biological predictive performance.


In [ ]:
show_source(
    reference.ReferenceCrossAttention,
    reference.ReferenceFusion,
    reference.LiveIsoFormer,
    full=SHOW_IMPLEMENTATION,
)

### Handling missing proteins

Retaining eligible transcripts without supplied proteins is agreed and the bypass is implemented. The [GTEx audit](tier2_isoformer_gtex_audit.ipynb#How-could-a-transcript-without-protein-enter-IsoFormer?) explains the affected population. The paper describes zeroing an absent modality's cross-attention contribution, while the released wrapper always executes its protein branch. Our bypass reconstructs that paper-described behavior; it is not recovered historical training code. [Paper, Section 4.2](https://proceedings.neurips.cc/paper_files/paper/2024/file/8f6b3692297e49e5d5c91ba00281379c-Paper-Conference.pdf) · [released wrapper](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py).

For a supplied protein within the length policy, keep the released computation. For an absent protein, skip that branch and pass **the RNA-updated DNA representation unchanged** to the same pooling and tissue head. The bypass removes the complete projected protein update, including bias; zero input embeddings would not guarantee this. Availability comes from the supplied field, not an assumed noncoding annotation.

The live wrapper takes an explicit per-row `protein_present` vector and protein inputs containing only those present rows, in their original order. It does not call the protein encoder for an all-absent batch. Synthetic tests check mixed batches, exact bypass, gradients and unchanged present-protein behavior. The future data loader must preserve this ordering and the selected per-example padding lengths; the complete real-data path has not been tested.


### Initialization is part of the experiment

A reproduction fit starts from **pretrained Enformer, NT and ESM-2 weights**, plus newly initialized fusion and prediction-head parameters. Fine-tuning can then update those encoders for GTEx. In contrast, evaluating the finished IsoFormer checkpoint measures a model that has already undergone its authors' task training; it cannot serve as a fresh independent reproduction run.

A model configuration specifies dimensions and structure, not learned weights. The released wrapper constructs NT/ESM from configurations and loads Enformer separately, so constructing it alone does not initialize all three from their original pretrained checkpoints. Our integration must load and verify each intended state explicitly. [Released constructor](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py).

The concrete adapters accept already constructed backbone modules; they do not download or load weights. No weight-loading path is implemented here. Before training, verify checkpoint/tokenizer identities, trainable parameters, actual sequence lengths, output shapes and pooling coordinates. These checks establish that the model receives the intended initialization and inputs; they do not reproduce a published score.


### Prepare the pretrained interfaces

An **adapter** connects a backbone's existing output to the fusion's expected tensor. `EnformerAdapter` requests cropped embeddings without changing the backbone's crop; `TokenEncoderAdapter` reads the final NT/ESM hidden state and preserves gradient connections. Both accept already constructed modules. They do not load weights, freeze parameters or create a trainer.

The DNA preparation starts at the released wrapper's post-CLS boundary: one base index per nucleotide, using A/C/G/T/N. RNA and protein tokenization retains the declared per-example padding floor without truncation. The supplied tokenizer must have the released special-token behavior; vocabulary identity still requires verification against the selected assets. [Released wrapper](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py) · [released tokenizer](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/isoformer_tokenizer.py).

If tokenized examples have unequal lengths after their individual padding, the preparation helper refuses that batch. Padding them further could change the unmasked protein fusion. This refusal preserves the open batching question; it does not impose another crop, remove long-but-eligible proteins, or define a training sampler. A future loader must keep protein inputs in the order selected by `protein_present`.


In [ ]:
# These calls display definitions only; no tokenizer or pretrained model is constructed.
show_source(
    adapters.dna_base_ids,
    adapters.tokenize_sequence_batch,
    adapters.EnformerAdapter,
    adapters.TokenEncoderAdapter,
    full=SHOW_IMPLEMENTATION,
)


---

## What is ready, and what remains?

**Implemented foundations.** The fusion operator, absent-protein bypass, in-memory sequence and cohort-eligibility policies, DNA-window helpers, validation helper, concrete backbone adapters and readiness guard exist. The [synthetic checks](../../tests/README.md#isoformer-reference-foundations) exercise these operations on invented tables and small substitute encoders. They do not establish which GTEx rows the authors trained on or verify real pretrained integration.

**Engineering still required.** Load and verify the original pretrained encoders/tokenizers and assets; verify the prepared cohort against the actual CSV/genome and finish target processing; connect a batching/training path that preserves the chosen inputs; then add checkpoint recovery and protected result writing. The single/subset models also need their own source-backed heads. GPU memory and runtime must be measured for that actual configuration.

**Choices still required.** Settle validation sampling, normalization/evaluation scale, and the remaining optimizer, precision and stopping settings. Confirm repeat counts and compute allocation before execution. The [specification](RUNBOOK.md#isoformer-reproduction-specification) keeps those decisions separate from settings already agreed.

The next table combines unset fields with missing integration. Filling those fields alone cannot create a trainer or establish pretrained compatibility.


In [ ]:
display(pd.DataFrame({
    "remaining prerequisite": training_blockers(choices)
}))
print("No embeddings, pretrained weights, caches, training or result writes were used.")

### What the eventual comparison must answer

The purpose of faithful reconstruction is to make the later comparisons interpretable: what each DNA representation predicts alone, what RNA/protein add, and what the fusion operator contributes. The [staged specification](RUNBOOK.md#staged-dna-backbone-comparisons) defines this progression.

| Stage | Question answered by matched modality comparisons |
| --- | --- |
| **Enformer first** | Do RNA and protein add predictive value to the released genomic-DNA reference? |
| **NT-DNA next** | Do those additions still help when a nucleotide language model supplies DNA representations? |
| **Borzoi later** | Does another DNA backbone and genomic context change how much RNA and protein contribute? |

Within each stage, **DNA only → DNA + RNA → DNA + RNA + protein** means three separately trained configurations and two incremental comparisons, not successive training steps in one model. RNA-only and protein-only baselines check whether the combination improves on the strongest relevant single representation. A gain over that single supports predictive complementarity; a gain over a matched simpler fusion, such as a separately specified concatenation control, tests the fusion method's added value. Neither identifies a biological mechanism from attention weights.

Match eligible rows, partitions, target processing and training/model-selection rules. For protein-only evaluation, compare every model on the same supplied-protein rows and report the broader cohort separately. Report protein-present and protein-absent groups and the common long-protein exclusion. Changing DNA backbone can also change context, eligibility and head capacity, so cross-backbone differences cannot be attributed to fusion alone. RNA-only or protein-only results may be reused only when their complete experiment identities match.

This notebook establishes an inspectable implementation route, not new biological findings. The complete seven-condition paper grid, alternative RNA/protein checkpoints and later DNA adapters remain extensions beyond the first configuration. Planning is tracked under `Gantt`/`Details` task 44 and uncertainty evaluation under task 28; the workbook's older pilot schedule is not a runtime estimate. The agreed comparison order does not authorize training.
